# 02 - Baselines B0 y B1

B0 predice la prevalencia estimada en `train`; B1 convierte una puntuación fija de recursos en probabilidad usando solo `train`. Los escenarios A, B y C se procesan por separado y cada partida tiene el mismo peso total. Este cuaderno no usa `calib` ni `test`.

La fórmula de B1 sigue siendo una propuesta pendiente de confirmación; sus resultados no deben declararse evidencia final hasta confirmar esa decisión.

La primera celda prepara el runtime de Colab y descarga las tablas procesadas si faltan.

In [ ]:
# Prepara el runtime de Colab y descarga las tablas procesadas si hacen falta.
from pathlib import Path
import gc, sys
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo pull
%cd /content/repo
!pip install -q -r requirements.txt
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')
# Evita que Python conserve una versión vieja del código después de actualizar el repo.
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
DATA_DIR = Path('/content/datos')
from ptcg_wp.ingest import load_days
dias = load_days('configs/dias.yaml')
# Archivos mínimos para modelar: tablas de estados por día y asignación de particiones.
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(ruta.exists() for ruta in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()  # Lee el token guardado en Colab o lo solicita.
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(ruta.exists() for ruta in esperados), 'Faltan tablas: revisa la descarga anterior.'

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from ptcg_wp.baselines import (ScenarioStream, weights_by_game, b0_and_hp_unit,
                               b1_probability, fit_b1_mapping)
from ptcg_wp.features import feature_columns
SCENARIOS = ('A', 'B', 'C')
print('Entorno listo; tablas disponibles:', len(list(DATA_DIR.glob('*.parquet'))), '| escenarios:', SCENARIOS)


## Peso igual por partida

Una partida con 200 estados da peso `1/200` a cada estado; una con 20 da `1/20`. Cada partida suma 1 en total. Para una métrica por etapa, los pesos se recalculan dentro de esa etapa.

In [ ]:
# Calcula AUC, log loss y Brier comparando etiquetas reales con probabilidades predichas.
def metricas(y, p, game_ids):
    w = weights_by_game(game_ids)
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-15, 1 - 1e-15)  # Evita probabilidades exactamente 0 o 1.
    return {
        'auc': roc_auc_score(y, p, sample_weight=w),
        'log_loss': log_loss(y, p, sample_weight=w, labels=[0, 1]),
        'brier': brier_score_loss(y, p, sample_weight=w),
        'n_estados': len(y),
        'n_partidas': pd.Series(game_ids).nunique(),
    }

# Calcula reportes para todos los estados y también por etapa de la partida.
def evaluar(val, pred, escenario, modelo):
    filas = []
    grupos = [('global', np.ones(len(val), dtype=bool))]
    grupos += [(etapa, val['stage'].to_numpy() == etapa) for etapa in ('inicio', 'medio', 'final')]
    for etapa, mascara in grupos:
        parte = val.loc[mascara]
        if len(parte):
            filas.append({'escenario': escenario, 'modelo': modelo, 'etapa': etapa, **metricas(parte.y, pred[mascara], parte.game_id)})  # ** añade las métricas al reporte.
    return filas
print('Funciones de ponderación y métricas listas.')


## B0 - probabilidad a priori

B0 ignora el tablero. Calcula en train la fracción de partidas ganadas por quien actúa y predice esa probabilidad en todos los estados de val. Es el piso de comparación.

## B1 - heurística de recursos

**Propuesta concreta para revisar:** `score = (opp_prizes - own_prizes) + diff_total_hp / hp_unit + diff_bench_n`. El signo queda positivo cuando el estado favorece al jugador que actúa. `hp_unit` será el promedio de `active_max_hp` positivo observado en los estados de `train`, ponderando para que cada partida tenga el mismo peso. Así, una ventaja en PV se expresa en unidades del HP promedio observado; no elegimos 100 sin evidencia. Este promedio se calcula por escenario y no usa `y` ni los IDs de carta.

Después, una regresión logística de una sola variable aprende en train la pendiente y el intercepto que convierten ese score en probabilidad. No aprende pesos separados para premios, PV y banca. Esta fórmula es una propuesta basada en recursos del juego; hay que revisarla antes de interpretar los resultados como evidencia final.

## Ejecución y registro en Drive

B1 sigue siendo una propuesta; los archivos guardados llevan esa condición. Ejecutar esta sección calcula B0 y B1 para A, B y C.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
import subprocess, json, time
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_DIR = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints/b0_b1') / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DIRTY = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
tiempos_escenario = {}
print('Resultados:', OUTPUT_DIR, '| commit:', COMMIT, '| cambios locales:', DIRTY)


In [ ]:
# Ejecuta B0 y B1 por escenario y guarda sus métricas y parámetros.
resultados, ajustes = [], []
columnas = ['game_id', 'y', 'stage', 'own_prizes', 'opp_prizes', 'own_active_max_hp', 'opp_active_max_hp', 'diff_total_hp', 'diff_bench_n']
columnas_parquet = [columna for columna in columnas if columna != 'stage']  # La etapa se deriva del turno al leer cada día.
for escenario in SCENARIOS:
    inicio_escenario = time.perf_counter()
    stream = ScenarioStream(DATA_DIR, escenario, tuple(columnas_parquet))  # Lee un día a la vez para controlar la RAM.
    prevalencia, hp_unit = b0_and_hp_unit(stream)
    beta_b1 = fit_b1_mapping(stream, hp_unit, prevalencia)

    partes_val = []
    for dia in stream:
        val = dia.loc[dia['split'] == 'val', ['game_id', 'y', 'stage']].copy()
        if len(val):
            mascara_val = dia['split'] == 'val'
            val['pred_b1'] = b1_probability(dia.loc[mascara_val], hp_unit, beta_b1)
            partes_val.append(val)
    val = pd.concat(partes_val, ignore_index=True)
    resultados += evaluar(val, np.full(len(val), prevalencia), escenario, 'B0')
    resultados += evaluar(val, val['pred_b1'].to_numpy(), escenario, 'B1')
    ajustes.append({'escenario': escenario, 'prevalencia_B0_train': prevalencia, 'hp_unit_train': hp_unit,
                    'intercepto_B1': beta_b1[0], 'pendiente_B1': beta_b1[1],
                    'partidas_val': val.game_id.nunique()})
    tiempos_escenario[escenario] = time.perf_counter() - inicio_escenario
    print(f'Escenario {escenario}: listo')
    del stream, partes_val, val, beta_b1
    gc.collect()  # Solicita liberar memoria antes de cargar el escenario siguiente.

metricas_baselines = pd.DataFrame(resultados)
ajustes_baselines = pd.DataFrame(ajustes)
display(metricas_baselines.round(4))
display(ajustes_baselines.round(4))
metricas_baselines.to_csv(OUTPUT_DIR / 'metricas_B0_B1.csv', index=False)
ajustes_baselines.to_csv(OUTPUT_DIR / 'ajustes_B0_B1.csv', index=False)
registro = {'commit': COMMIT, 'working_tree_dirty': DIRTY, 'run_id_utc': RUN_ID,
            'seed': 2026, 'escenarios': list(SCENARIOS), 'split_fit': 'train', 'split_eval': 'val',
            'ponderacion': 'cada partida suma 1 en cada subconjunto',
            'B1_formula': '(opp_prizes - own_prizes) + diff_total_hp / hp_unit + diff_bench_n',
            'B1_estado': 'propuesta_pendiente_confirmacion', 'segundos_por_escenario': tiempos_escenario}
(OUTPUT_DIR / 'registro.json').write_text(json.dumps(registro, indent=2, ensure_ascii=False))
print('Resultados guardados en Drive:', OUTPUT_DIR)


## Descargar los resultados

Colab no descarga carpetas desde el panel como una carpeta normal. Esta celda comprime los tres archivos de la corrida en un ZIP y lanza la descarga al equipo. Ejecútala después de terminar B0/B1.

In [ ]:
from pathlib import Path
import shutil
from google.colab import files
assert OUTPUT_DIR.exists() and list(OUTPUT_DIR.iterdir()), 'Primero ejecuta B0/B1 y guarda los resultados en Drive.'
zip_base = Path('/content') / f'resultados_B0_B1_{RUN_ID}'
zip_file = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR))
print('Descargando:', zip_file, '| bytes:', zip_file.stat().st_size)
files.download(str(zip_file))
